# BTLON02 — Notebook 01: Khám phá dữ liệu (EDA) & Data Augmentation (FER2013)
- **Phụ trách**: Thành viên 1 — Lý Hồng Hưng (`3123411132`)
- **Môi trường khuyến nghị**: Kaggle Notebook (Add Input: `deadskull7/fer2013` hoặc `msambare/fer2013`, bật `Internet: On`)
- **Mục tiêu**: Thống kê phân bố 7 lớp cảm xúc, chỉ ra mất cân bằng lớp `Disgust`, xuất ảnh mẫu 4×8 và minh họa Data Augmentation ở độ phân giải 300 DPI cho Report & Slide.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/HongHungLy/SGU26K2_HocSau_S5_DCT123C4_Nhom666HocSau.git"
BRANCH = "develop"
PROJECT_SUBDIR = "BTLON02_Emotion_Ly Hong Hung_3123411132"

if Path("/kaggle/working").exists():
    repo_dir = Path("/kaggle/working/repo")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull", "origin", BRANCH], check=True)
    PROJECT_DIR = repo_dir / PROJECT_SUBDIR
else:
    # Chạy local từ thư mục notebooks/ hoặc thư mục gốc dự án
    cwd = Path.cwd()
    PROJECT_DIR = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Working directory:", Path.cwd())

## 1. Kiểm tra nguồn dữ liệu FER2013 và nạp DataLoader

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

from src.dataset import discover_fer2013_source, get_dataloaders, get_transforms
from src.utils import EMOTION_LABELS, ensure_dir, set_seed

set_seed(42)
ensure_dir("figures")
ensure_dir("logs")

source = discover_fer2013_source()
print("Đã tìm thấy nguồn dữ liệu:", source)

train_loader_raw, val_loader, test_loader, info = get_dataloaders(
    batch_size=64, use_aug=False, img_size=48, in_channels=1, num_workers=2, seed=42
)
train_loader_aug, _, _, _ = get_dataloaders(
    batch_size=64, use_aug=True, img_size=48, in_channels=1, num_workers=2, seed=42
)

print(f"Training:   {info['train_size']:,} ảnh")
print(f"PublicTest: {info['val_size']:,} ảnh (Validation)")
print(f"PrivateTest:{info['test_size']:,} ảnh (Test độc lập)")
print(f"Tổng cộng:  {info['train_size'] + info['val_size'] + info['test_size']:,} ảnh")

## 2. Bảng thống kê & Biểu đồ phân bố 7 lớp cảm xúc (Histogram)

In [ ]:
def count_per_class(labels):
    return [int((labels == i).sum()) for i in range(len(EMOTION_LABELS))]

tr_cnt = count_per_class(info["train_labels"])
va_cnt = count_per_class(info["val_labels"])
te_cnt = count_per_class(info["test_labels"])

df_dist = pd.DataFrame({
    "Label_ID": list(range(7)),
    "Emotion": EMOTION_LABELS,
    "Training": tr_cnt,
    "PublicTest_Val": va_cnt,
    "PrivateTest_Test": te_cnt,
})
df_dist["Total"] = df_dist["Training"] + df_dist["PublicTest_Val"] + df_dist["PrivateTest_Test"]
df_dist["Train_Ratio_%"] = (df_dist["Training"] / df_dist["Training"].sum() * 100).round(2)

df_dist.to_csv("logs/eda_class_distribution.csv", index=False, encoding="utf-8-sig")
display(df_dist)

imbalance_ratio = max(tr_cnt) / max(min(tr_cnt), 1)
print(f"Tỉ lệ mất cân bằng lớn nhất trên Training (Happy / Disgust): {imbalance_ratio:.2f}x")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
splits = [
    ("Training Dataset", tr_cnt, "#1B365D"),
    ("PublicTest (Validation)", va_cnt, "#2B5B84"),
    ("PrivateTest (Testing)", te_cnt, "#4A7C9B"),
]

for ax, (title, counts, color) in zip(axes, splits):
    colors = ["#C00000" if i == 1 else color for i in range(7)]  # Tô đỏ lớp Disgust thiểu số
    bars = ax.bar(EMOTION_LABELS, counts, color=colors, edgecolor="black", alpha=0.9)
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{int(h)}", xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9)
    ax.set_title(f"{title} (N = {sum(counts):,})", fontsize=12, fontweight="bold")
    ax.set_ylabel("Số lượng ảnh")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Phân bố 7 lớp cảm xúc trên bộ dữ liệu Kaggle FER2013 (Đỏ: lớp thiểu số Disgust)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/eda_histogram_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Trực quan hóa lưới ảnh mẫu 4×8 (Training, Validation & Sau Augmentation)

In [ ]:
def plot_image_grid(loader, title, save_path, rows=4, cols=8):
    images, labels = next(iter(loader))
    n = min(rows * cols, len(images))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.8))
    for i, ax in enumerate(axes.flat):
        if i < n:
            img = images[i].squeeze().numpy() * 0.5 + 0.5  # Giải chuẩn hóa [-1, 1] -> [0, 1]
            ax.imshow(np.clip(img, 0, 1), cmap="gray")
            ax.set_title(EMOTION_LABELS[int(labels[i])], fontsize=9, fontweight="bold")
        ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

plot_image_grid(train_loader_raw, "Ảnh mẫu tập Training (48x48 Grayscale)", "figures/eda_train_samples.png")
plot_image_grid(val_loader, "Ảnh mẫu tập PublicTest / Validation", "figures/eda_val_samples.png")
plot_image_grid(train_loader_aug, "Ảnh mẫu tập Training sau Data Augmentation", "figures/eda_augmented_batch.png")

## 4. Minh họa trực quan 1 ảnh gốc vs 6 biến thể Data Augmentation

In [ ]:
raw_ds = train_loader_raw.dataset
aug_tf = get_transforms(is_train=True, use_aug=True, img_size=48, in_channels=1)

# Lấy 1 ảnh gốc rõ nét (Happy)
sample_idx = int(np.where(info["train_labels"] == 3)[0][0])
img_tensor, lbl = raw_ds[sample_idx]
orig_np = np.clip((img_tensor.squeeze().numpy() * 0.5 + 0.5) * 255.0, 0, 255).astype(np.uint8)
pil_img = Image.fromarray(orig_np, mode="L")

fig, axes = plt.subplots(1, 7, figsize=(15, 2.6))
axes[0].imshow(orig_np, cmap="gray")
axes[0].set_title("Ảnh gốc\n(Original)", fontsize=10, fontweight="bold", color="#1B365D")
axes[0].axis("off")

for k in range(1, 7):
    aug_t = aug_tf(pil_img)
    aug_np = np.clip(aug_t.squeeze().numpy() * 0.5 + 0.5, 0, 1)
    axes[k].imshow(aug_np, cmap="gray")
    axes[k].set_title(f"Aug #{k}", fontsize=10)
    axes[k].axis("off")

fig.suptitle(f"Minh họa Data Augmentation trên cùng 1 mẫu (Nhãn: {EMOTION_LABELS[lbl]})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/eda_augmentation_demo.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Đóng gói kết quả EDA thành file ZIP để tải về từ Kaggle

In [ ]:
import zipfile

zip_target = Path("/kaggle/working/tv1_eda_outputs.zip") if Path("/kaggle/working").exists() else Path("tv1_eda_outputs.zip")
with zipfile.ZipFile(zip_target, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["figures", "logs"]:
        for f in Path(folder).rglob("*"):
            if f.is_file() and f.name != ".gitkeep":
                zf.write(f, arcname=str(f))

print("Đã nén xong kết quả EDA tại:", zip_target)
print("Hãy tải file này về máy, giải nén vào BTLON02_Emotion_Ly Hong Hung_3123411132/ rồi push lên nhánh develop!")